In [30]:
import cosmic
from cosmic.output import load_initC
import h5py as h5
import os
import pandas as pd

In [46]:
DCO_TYPES = ["NSWD", "NSNS", "BHWD", "BHNS", "BHBH"]
kstar_masks = [([13], [10, 11, 12]), ([13], [13]), ([14], [10, 11, 12]), ([14], [13]), ([14], [14])]
FOLDER = "/mnt/ceph/users/twagg/lisa-dcos/fiducial"

for kstars, dco_type in zip(kstar_masks, DCO_TYPES):
    if dco_type == "BHWD":
        continue
    print(f"Processing {dco_type}...")
    all_formation_rows = None
    for file in os.listdir(FOLDER):
        if file.startswith(f"{dco_type}_Z_"):
            print(f"  Adding file: {file}")
            initC = cosmic.output.load_initC(f"{FOLDER}/{file}", key="initC")
            bpp = pd.read_hdf(f"{FOLDER}/{file}", key="bpp")
            k_lo, k_hi = kstars
            mask = ((
                (bpp["kstar_1"].isin(k_lo)) & (bpp["kstar_2"].isin(k_hi))
            ) | (
                (bpp["kstar_1"].isin(k_hi)) & (bpp["kstar_2"].isin(k_lo))
            )) & (bpp["sep"] > 0)
            if mask.sum() == 0:
                print(f"    No {dco_type} systems found in this file.")
                continue
            formation_rows = bpp[mask].drop_duplicates(subset="bin_num")[["tphys", "mass_1", "mass_2", "kstar_1", "kstar_2", "sep", "ecc", "bin_num"]]
            formation_rows["metallicity"] = initC["metallicity"].iloc[0]

            with h5.File(f"{FOLDER}/{file}", "r") as f:
                formation_rows["weights"] = f["stroopwafel"]["weights"][...][f["stroopwafel"]["is_hit"][...]]

            if all_formation_rows is None:
                all_formation_rows = formation_rows
            else:
                all_formation_rows = pd.concat([all_formation_rows, formation_rows], ignore_index=True)
    all_formation_rows.to_hdf(f"{FOLDER}/{dco_type}_formation_rows.h5", key="formation_rows", mode="w")

Processing NSWD...
  Adding file: NSWD_Z_0.0001.h5
  Adding file: NSWD_Z_0.01883.h5
  Adding file: NSWD_Z_0.03.h5
  Adding file: NSWD_Z_0.00329.h5
Processing NSNS...
  Adding file: NSNS_Z_0.01883.h5
  Adding file: NSNS_Z_0.03.h5
  Adding file: NSNS_Z_0.00329.h5
  Adding file: NSNS_Z_0.0001.h5
Processing BHNS...
  Adding file: BHNS_Z_0.00329.h5
  Adding file: BHNS_Z_0.0001.h5
  Adding file: BHNS_Z_0.01883.h5
  Adding file: BHNS_Z_0.03.h5
    No BHNS systems found in this file.
Processing BHBH...
  Adding file: BHBH_Z_0.0023170977998889395.h5
  Adding file: BHBH_Z_0.0066058060708354014.h5
  Adding file: BHBH_Z_0.001835850765193497.h5
  Adding file: BHBH_Z_0.02670347368991138.h5
    No BHBH systems found in this file.
  Adding file: BHBH_Z_0.00028508965271781034.h5
  Adding file: BHBH_Z_0.029999999999999995.h5
    No BHBH systems found in this file.
  Adding file: BHBH_Z_0.0014545557948495277.h5
  Adding file: BHBH_Z_0.01676307853735573.h5
  Adding file: BHBH_Z_0.009366693401651872.h5
  A

In [ ]:
all_formation_rows

In [51]:
all_formation_rows

,tphys,mass_1,mass_2,kstar_1,kstar_2,sep,ecc,bin_num,metallicity,weights
0,9.674936,11.481352,10.022770,14,14,7.955414,0.069387,131,0.002317,0.058596
1,11.519324,4.266478,7.354409,14,14,8.704415,0.628392,142,0.002317,0.141956
2,10.910525,9.802658,6.588742,14,14,5.330900,0.186542,244,0.002317,0.089382
3,13.530594,4.656871,4.802118,14,14,6.112695,0.520843,257,0.002317,0.150346
4,8.397477,14.869530,15.085512,14,14,12.741032,0.016692,413,0.002317,0.035048
...,...,...,...,...,...,...,...,...,...,...
3207082,5.843718,27.617264,37.143314,14,14,9.730228,0.007721,499978,0.000254,0.063306
3207083,6.433255,22.437180,31.668513,14,14,6.529553,0.009241,499979,0.000254,0.063451
3207084,6.968144,22.276963,26.772448,14,14,5.913335,0.010194,499984,0.000254,0.045237
3207085,6.679448,22.316637,28.583285,14,14,6.505550,0.009823,499987,0.000254,0.035607
